# Beam Noise Diffusion (BND) Experiment

This notebook clones the latest code from GitHub into the Colab runtime. Google Drive is used only for persistent runtime artifacts: Hugging Face cache, generated images, result JSON files, and CSV summaries.


## Step 0: Check GPU

Verify which GPU Colab assigned before loading the diffusion model.


In [8]:
!nvidia-smi

Sat May 23 20:11:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             51W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Step 1: Mount Google Drive

Drive is used only as persistent storage for `cache/` and `outputs/`.


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Step 2: Clone Code from GitHub

The repository is cloned into the temporary Colab runtime. This keeps executable code separate from persistent outputs.


In [10]:
from pathlib import Path
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/nguyenphamngocquy/beam-noise-diffusion.git"
CODE_ROOT = Path("/content/beam-noise-diffusion")
STORAGE_ROOT = Path("/content/drive/MyDrive/beam-noise-diffusion-storage")

if CODE_ROOT.exists():
    shutil.rmtree(CODE_ROOT)

subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CODE_ROOT)], check=True)
sys.path.insert(0, str(CODE_ROOT))

print("Code root:", CODE_ROOT)
print("Storage root:", STORAGE_ROOT)

Code root: /content/beam-noise-diffusion
Storage root: /content/drive/MyDrive/beam-noise-diffusion-storage


## Step 3: Install Dependencies

Install dependencies from the cloned repository's `requirements.txt`.


In [11]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "-r",
    str(CODE_ROOT / "requirements.txt"),
], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-r', '/content/beam-noise-diffusion/requirements.txt'], returncode=0)

## Step 4: Load Config and Prompts

Configs and prompts are read from the cloned repository. Runtime outputs are written to `STORAGE_ROOT` on Drive.


In [12]:
from bnd.config import load_config, load_prompts
from bnd.io import ensure_experiment_dirs

CONFIG_NAME = "bnd_full"  # Change to "bnd_full" for A100/full run
ensure_experiment_dirs(STORAGE_ROOT)
CFG = load_config(CODE_ROOT, CONFIG_NAME)
PROMPTS = load_prompts(CODE_ROOT)
RUN_NAME = CFG["name"]

print("Config:", RUN_NAME)
print("Perturbation:", CFG.get("perturbation_mode", "additive"))
print("Number of prompts:", len(PROMPTS))
print(PROMPTS[0])

Config: bnd_full
Perturbation: prior_mix
Number of prompts: 30
{'id': '0001', 'group': 'simple', 'prompt': 'a photo of a golden retriever sitting in a park'}


In [13]:
# Load HF_TOKEN from user input
import os
from getpass import getpass

token = getpass("Enter HF_TOKEN: ").strip()
os.environ["HF_TOKEN"] = token

print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
# Verify HF_TOKEN by calling whoami
from huggingface_hub import whoami

print(whoami(token=os.environ["HF_TOKEN"]))

## Step 5: Load Models

Stable Diffusion and CLIP are loaded from code in the cloned repo. Model files are cached on Drive under `STORAGE_ROOT/cache/hf_cache`.


In [14]:
from bnd.sampling import get_gpu_name, load_sd_pipeline
from bnd.scoring import ClipScorer

print("GPU:", get_gpu_name())
CACHE_DIR = STORAGE_ROOT / "cache" / "hf_cache"
pipe = load_sd_pipeline(CFG, CACHE_DIR)
scorer = ClipScorer(CACHE_DIR)

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


GPU: NVIDIA A100-SXM4-80GB


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /content/drive/MyDrive/beam-noise-diffusion-storage/cache/hf_cache/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


xFormers enabled.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/open_clip/factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


## Step 6: Run Experiments

For each prompt, the notebook runs three methods: random baseline, Monte-Carlo selection, and BND. Set `PROMPT_LIMIT` to a small number for a smoke test.


In [15]:
from bnd.algorithm import run_bnd, run_monte_carlo, run_random_baseline

PROMPT_LIMIT = None  # Set to a small number, e.g. 2, for a quick smoke test
items = PROMPTS if PROMPT_LIMIT is None else PROMPTS[:PROMPT_LIMIT]

for item in items:
    run_random_baseline(item, CFG, pipe, scorer, STORAGE_ROOT, RUN_NAME)
    run_monte_carlo(item, CFG, pipe, scorer, STORAGE_ROOT, RUN_NAME)
    run_bnd(item, CFG, pipe, scorer, STORAGE_ROOT, RUN_NAME)

  0%|          | 0/30 [00:00<?, ?it/s]

[DONE] random 0001 score=0.3353


  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

...

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/12 [00:00<?, ?it/s]

  0%|          | 0/30 [00:00<?, ?it/s]

[DONE] bnd 0030 score=0.3885


## Step 7: Summarize Results

Collect JSON result files from Drive into CSV summaries and display aggregate metrics.


In [16]:
from bnd.io import summarize_results

df, summary, group_summary = summarize_results(STORAGE_ROOT, RUN_NAME)
display(df.head())
display(summary)
display(group_summary)

,prompt_id,prompt,group,method,score,runtime_sec,gpu,image_path
0,0001,a photo of a golden retriever sitting in a park,simple,random,0.335283,3.793603,NVIDIA A100-SXM4-80GB,/content/drive/MyDrive/beam-noise-diffusion-st...
1,0002,a red apple on a wooden table,simple,random,0.290817,1.895266,NVIDIA A100-SXM4-80GB,/content/drive/MyDrive/beam-noise-diffusion-st...
2,0003,a small boat floating on a calm lake,simple,random,0.313035,1.893738,NVIDIA A100-SXM4-80GB,/content/drive/MyDrive/beam-noise-diffusion-st...
3,0004,a white cat sleeping on a sofa,simple,random,0.331496,1.918958,NVIDIA A100-SXM4-80GB,/content/drive/MyDrive/beam-noise-diffusion-st...
4,0005,a mountain landscape during sunset,simple,random,0.309809,1.938173,NVIDIA A100-SXM4-80GB,/content/drive/MyDrive/beam-noise-diffusion-st...


,method,mean_clipscore,std_clipscore,mean_runtime,count
0,bnd,0.346468,0.031519,15.223379,30
1,monte_carlo,0.341343,0.030817,8.018092,30
2,random,0.317242,0.029293,1.982156,30


,group,method,mean_clipscore,mean_runtime,count
0,attributes,bnd,0.359448,15.172171,10
1,attributes,monte_carlo,0.355918,7.985879,10
2,attributes,random,0.330070,1.924365,10
3,complex,bnd,0.352672,15.404153,10
4,complex,monte_carlo,0.341182,7.998784,10
5,complex,random,0.313090,1.917689,10
6,simple,bnd,0.327284,15.093814,10
7,simple,monte_carlo,0.326928,8.069614,10
8,simple,random,0.308565,2.104413,10


In [17]:
try:
    from google.colab import runtime
    runtime.unassign()
except ImportError:
    print("Not running in Google Colab; skipping runtime disconnect.")